In [ ]:
# =====================================================================
# Three-Way NSGA-II Comparison
#   Arm 1: Pure Physics NSGA-II (200 gen) — baseline ground truth
#   Arm 2: Pure LGBM Surrogate (200 gen) → physics-verified
#   Arm 3: Dual-Stage (LGBM 200 gen + Physics warm-start 20 gen)
# =====================================================================

import torch, numpy as np, math, time, json, glob, os
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import lightgbm as lgb
from pymoo.core.problem import ElementwiseProblem
from pymoo.core.callback import Callback
from pymoo.core.population import Population
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

try:
    device = torch.device('cuda'); torch.zeros(1,device=device)
except (AssertionError,RuntimeError): device=torch.device('cpu')
print(f'Device: {device}')

# ============================================================
# 1. Physics Engine
# ============================================================
rx,ry,rz=10.0,10.0,3.0; xBs,yBs,zBs=10.0,-100.0,-10.0
E=8; dB=0.075; lam=0.075; L1=2; dref=abs(yBs)*1.5
Pd=40.0; Rth=0.2; Nd=-174.0; Bw=20e6; pm_val=0.2; nu=5
PB=10**(Pd/10)*1e-3; N0=10**(Nd/10)*1e-3*Bw

def gen_rwp_traj(sim_time,dt=10,nu=5,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot'else tau_n
    def sp(r):return v_h if r=='hot'else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot'if d2h<=hr else'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

GR=200; Z_HS=[1.5,1.625,1.75,1.875,2.0]; N_Z=len(Z_HS)
xv=torch.linspace(0,rx,GR,device=device); yv=torch.linspace(0,ry,GR,device=device)
Xg,Yg=torch.meshgrid(xv,yv,indexing='ij'); xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
gl=[]; [gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1)) for zh in Z_HS]
gl=torch.cat(gl,dim=0); Ngrid=gl.shape[0]
np.random.seed(777); et=gen_rwp_traj(sim_time=100000,dt=10)
kde=gaussian_kde(et[:,:2].T,bw_method='scott')
wxy=kde(xyf.cpu().numpy().T).flatten(); wxy=wxy/wxy.sum()
gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device); gw=gw/gw.sum()
np.random.seed(42)
_ps=torch.tensor(2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ngrid),dtype=torch.float32,device=device)
_eta=torch.tensor(10**((-15+5*np.random.rand(Ngrid))/10),dtype=torch.float32,device=device)
_n_ant=torch.arange(E,dtype=torch.float32,device=device)
_dy_BS=torch.tensor(0.0-yBs,device=device); _v1c=lam/(4*math.pi); _v1pc=-(2*math.pi/lam); _oE=1/math.sqrt(E)

def phys_chan(wp,lc):
    if not isinstance(wp,torch.Tensor): wp=torch.tensor(wp,dtype=torch.float32,device=device)
    if wp.dim()==1: wp=wp.unsqueeze(0)
    Bn=wp.shape[0]; xc,zc,Lx,Lz=wp[:,0],wp[:,1],wp[:,2],wp[:,3]
    xu,yu,zu=lc[:,0],lc[:,1],lc[:,2]
    dx=xc-xBs; dy=_dy_BS; dz=zc-zBs
    Rw=torch.sqrt(dx**2+dy**2+dz**2); th=torch.atan2(dy,dx); ph=torch.acos(dz/Rw)
    kx=torch.sin(ph)*torch.cos(th); kz=torch.cos(ph)
    x1=xc-Lx/2;z1=zc-Lz/2;x2=xc-Lx/2;z2=zc+Lz/2;x3=xc+Lx/2;z3=zc-Lz/2;x4=xc+Lx/2;z4=zc+Lz/2
    def rd(xs,zs):
        ddx=xs.unsqueeze(1)-xBs;ddy=_dy_BS;ddz=zs.unsqueeze(1)-zBs
        L=torch.sqrt(ddx**2+ddy**2+ddz**2);return ddx/L,ddy/L,ddz/L
    ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
    dux=xu-xBs;duy=yu-yBs;duz=zu-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2)
    uux=dux/Lu;uuz=duz/Lu
    ua=torch.stack([ux1,ux2,ux3,ux4],0);uz=torch.stack([uz1,uz2,uz3,uz4],0)
    umin=ua.min(0).values;umax=ua.max(0).values;zmin=uz.min(0).values;zmax=uz.max(0).values
    b=1000.0
    ix=torch.sigmoid(b*(uux-umin))*torch.sigmoid(b*(umax-uux))
    iz=torch.sigmoid(b*(uuz-zmin))*torch.sigmoid(b*(zmax-uuz))
    il=ix*iz
    d2x=xu-xc.unsqueeze(1);d2y=yu;d2z=zu-zc.unsqueeze(1)
    Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
    ax=(1-il)*(kx.unsqueeze(1)-t1);az=(1-il)*(kz.unsqueeze(1)-t2)
    sx=torch.sinc((math.pi/lam)*Lx.unsqueeze(1)*ax);sz=torch.sinc((math.pi/lam)*Lz.unsqueeze(1)*az)
    p1=(2*math.pi/lam)*dB*_n_ant*torch.sin(th).unsqueeze(1)
    a1=_oE*torch.complex(torch.cos(p1),torch.sin(p1))
    v1m=_v1c/Rw;v1p=_v1pc*Rw
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1.unsqueeze(1)*a1.conj()
    p2=(2*math.pi/lam)*dB*_n_ant.unsqueeze(0)*torch.sin(_tt).unsqueeze(1)
    a2=_oE*torch.complex(torch.cos(p2),torch.sin(p2))
    v2m=_eta*(lam/(4*math.pi*dref));v2=torch.complex(v2m*torch.cos(_ps),v2m*torch.sin(_ps))
    H2=v2.unsqueeze(1)*a2.conj().unsqueeze(0)
    Ht=math.sqrt(E/L1)*(H1.unsqueeze(1)+H2)
    fm=(Lx.unsqueeze(1)*Lz.unsqueeze(1)*sx*sz)/(lam*Ru)
    fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2))
    fc=torch.complex(fm*torch.cos(fp.unsqueeze(1)),fm*torch.sin(fp.unsqueeze(1)))
    return Ht*fc.unsqueeze(2)

@torch.no_grad()
def physics_outage(X_np):
    out=[]
    for i in range(0,len(X_np),200):
        xb=X_np[i:i+200]; th=torch.tensor(xb,dtype=torch.float32,device=device)
        He=phys_chan(th,gl); Hw=torch.sum(He,dim=2)/math.sqrt(E)
        dp=(torch.abs(Hw)**2)*pm_val*PB; it=(nu-1)*dp; sr=dp/(it+N0)
        xc_,zc_=th[:,0],th[:,1]; ab=math.pi/3.0; Ses=torch.zeros_like(sr)
        rn=torch.sqrt((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))**2+(gl[:,2].unsqueeze(0)-zc_.unsqueeze(1))**2+gl[:,1].unsqueeze(0)**2+1e-12)
        for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
            dot=torch.abs((gl[:,0].unsqueeze(0)-xc_.unsqueeze(1))*math.cos(a)+gl[:,1].unsqueeze(0)*math.sin(a))
            cp=dot/rn; ph=torch.acos(torch.clamp(cp,0,1))
            Se=(math.pi-torch.clamp(2*ab-ph,min=0))/math.pi; Ses=Ses+torch.clamp(Se,1/3,1)
        sr=((Ses/4)*dp)/((Ses/4)*it+N0)
        bits=(torch.log2(1+sr)<Rth).float()
        out.append(torch.sum(bits*gw,dim=1).cpu().numpy()); torch.cuda.empty_cache()
    return np.concatenate(out)

print('Physics engine ready.')

# ============================================================
# 2. Load LGBM Surrogate
# ============================================================
model_path=None
for c in ['lgbm_surrogate.txt','/kaggle/working/lgbm_surrogate.txt']+sorted(glob.glob('/kaggle/input/**/lgbm_surrogate.txt',recursive=True)):
    if os.path.exists(c): model_path=c;break
if model_path is None: raise FileNotFoundError('lgbm_surrogate.txt')
lgbm_model=lgb.Booster(model_file=model_path)
print(f'LGBM loaded: {model_path}')

lb=np.array([0.2,0.2,0.1,0.1]); ub=np.array([9.8,2.8,9.8,2.8])

# ============================================================
# 3. Callback
# ============================================================
class Tracker(Callback):
    def __init__(self):
        super().__init__()
        self.data={'gen':[],'X_std':[],'viol_rate':[],'F':[],'X':[]}
    def notify(self,algorithm):
        pop=algorithm.pop; X=pop.get('X')
        self.data['gen'].append(algorithm.n_gen)
        self.data['X_std'].append(X.std(axis=0).tolist())
        xc,zc,Lx,Lz=X[:,0],X[:,1],X[:,2],X[:,3]
        viol=(Lx/2>xc)|(xc+Lx/2>10)|(Lz/2>zc)|(zc+Lz/2>3)
        self.data['viol_rate'].append(viol.mean())
        if algorithm.n_gen in [0,5,10,15,20]:
            self.data['F'].append((algorithm.n_gen,pop.get('F').copy()))
            self.data['X'].append((algorithm.n_gen,X.copy()))

class PhysicsProblem(ElementwiseProblem):
    def __init__(self): super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
    def _evaluate(self,x,out,*a,**k):
        xc,zc,Lx,Lz=x
        out["G"]=[Lx/2-xc,xc+Lx/2-10,Lz/2-zc,zc+Lz/2-3]
        out["F"]=[Lx*Lz,float(physics_outage(x.reshape(1,-1))[0])]

class SurrogateProblem(ElementwiseProblem):
    def __init__(self): super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
    def _evaluate(self,x,out,*a,**k):
        xc,zc,Lx,Lz=x
        out["G"]=[Lx/2-xc,xc+Lx/2-10,Lz/2-zc,zc+Lz/2-3]
        out["F"]=[Lx*Lz,float(lgbm_model.predict(x.reshape(1,-1))[0])]

n_gen_pure=200; n_gen_warm=20

# ============================================================
# ARM 1: Pure Physics NSGA-II
# ============================================================
print('\n'+'='*50)
print('ARM 1: Pure Physics NSGA-II (200 gen)')
print('='*50)
tracker_pure=Tracker()
algo1=NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True)
t0=time.time()
res_pure=minimize(PhysicsProblem(),algo1,get_termination('n_gen',n_gen_pure),seed=42,callback=tracker_pure,verbose=True)
t_pure=time.time()-t0
F_pure=res_pure.F; X_pure=res_pure.X
print(f'Pure NSGA-II: {t_pure:.0f}s, {len(F_pure)} pts')

# ============================================================
# ARM 2: Pure LGBM Surrogate (200 gen) → physics-verify
# ============================================================
print('\n'+'='*50)
print('ARM 2: Pure LGBM Surrogate NSGA-II (200 gen)')
print('='*50)
tracker_surr=Tracker()
t0=time.time()
res_surr_only=minimize(SurrogateProblem(),
    NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),
          crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True),
    get_termination('n_gen',n_gen_pure),seed=42,callback=tracker_surr,verbose=True)
t_surr_only=time.time()-t0
# Physics-verify the surrogate-only Pareto front
X_surr_only=res_surr_only.X
print(f'Physics-verifying {len(X_surr_only)} surrogate-only candidates...')
F_surr_only=np.column_stack([X_surr_only[:,2]*X_surr_only[:,3], physics_outage(X_surr_only)])
print(f'Pure surrogate: {t_surr_only:.0f}s, {len(F_surr_only)} pts')

# ============================================================
# ARM 3: Dual-Stage (LGBM 200 gen + Warm-Start 20 gen)
# ============================================================
print('\n'+'='*50)
print('ARM 3: Dual-Stage (LGBM 200 gen + Warm-Start 20 gen)')
print('='*50)
t0=time.time()
res_surr=minimize(SurrogateProblem(),
    NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),
          crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True),
    get_termination('n_gen',n_gen_pure),seed=42,verbose=True)
t_phase1=time.time()-t0

F_s=res_surr.F; mask=F_s[:,1]<0.30
warm_X=res_surr.X[mask][:250]
np.random.seed(99)
rand_X=np.random.uniform(low=lb,high=ub,size=(50,4))
hybrid_X=np.vstack([warm_X,rand_X])
warm_pop=Population.new("X",hybrid_X)

tracker_refine=Tracker()
t1=time.time()
res_refine=minimize(PhysicsProblem(),
    NSGA2(pop_size=300,sampling=warm_pop,
          crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.35,eta=15),eliminate_duplicates=True),
    get_termination('n_gen',n_gen_warm),seed=42,callback=tracker_refine,verbose=True)
t_refine=time.time()-t1
t_dual=t_phase1+t_refine
F_refine=res_refine.F; X_refine=res_refine.X
print(f'Dual-Stage: {t_dual:.0f}s ({t_phase1:.0f} LGBM + {t_refine:.0f} physics), {len(F_refine)} pts')

# ============================================================
# FIGURE 1 — Angle 1: Gene Diversity
# ============================================================
fig1,axes1=plt.subplots(2,2,figsize=(12,8))
labels_var=['xc','zc','Lx','Lz']
for i,(ax,lab) in enumerate(zip(axes1.flat,labels_var)):
    gen_p=tracker_pure.data['gen']
    ax.plot(gen_p,[s[i] for s in tracker_pure.data['X_std']],'gray',linewidth=1.5,alpha=0.7,label='Pure Physics')
    gen_s=tracker_surr.data['gen']
    ax.plot(gen_s,[s[i] for s in tracker_surr.data['X_std']],'red',linewidth=1.5,alpha=0.7,label='Pure Surrogate')
    gen_r=tracker_refine.data['gen']
    ax.plot(gen_r,[s[i] for s in tracker_refine.data['X_std']],'green',linewidth=2,label='Dual-Stage warm-start')
    ax.set_xlabel('Generation');ax.set_ylabel(f'std({lab})');ax.set_title(f'{lab} diversity')
    ax.legend(fontsize=7);ax.grid(alpha=0.3)
plt.suptitle('Angle 1: Gene Diversity — Pure vs Surrogate vs Dual-Stage',fontsize=14)
plt.tight_layout();plt.savefig('angle1_diversity.png',dpi=150);plt.show()

# ============================================================
# FIGURE 2 — Angle 2: Rebound (population drift under physics refinement)
snap_F=tracker_refine.data["F"];snap_X=tracker_refine.data["X"]
n_snap=len(snap_F)
fig2,axes2=plt.subplots(1,n_snap,figsize=(4*n_snap,4))
if n_snap==1:axes2=[axes2]
for ax,(gen,F_snap),(_,X_snap) in zip(axes2,snap_F,snap_X):
    phys_out=physics_outage(X_snap)
    areas=X_snap[:,2]*X_snap[:,3]
    ax.scatter(phys_out*100,areas,c="steelblue",s=8,alpha=0.4)
    ax.scatter(F_snap[:,1]*100,X_snap[:,2]*X_snap[:,3],c="red",s=4,alpha=0.3,marker="x")
    ax.axvline(x=10,color="red",ls="--",alpha=0.6)
    ax.set_xlim(0,50);ax.set_xlabel("Physics Outage [%]");ax.set_ylabel("Area [m2]")
    ax.set_title(f"Gen {gen}");ax.grid(alpha=0.3)
plt.suptitle("Angle 2: Rebound — Surrogate False Optima Drift Under Physics",fontsize=14)
plt.tight_layout();plt.savefig("angle2_rebound.png",dpi=150);plt.show()

# ============================================================
snap_F=tracker_refine.data['F'];snap_X=tracker_refine.data['X']
n_snap=len(snap_F)
fig3,axes3=plt.subplots(1,n_snap,figsize=(4*n_snap,4))
if n_snap==1:axes3=[axes3]
for ax,(gen,F_snap),(_,X_snap) in zip(axes3,snap_F,snap_X):
    phys_out=physics_outage(X_snap)
    areas=X_snap[:,2]*X_snap[:,3]
    ax.scatter(phys_out*100,areas,c='steelblue',s=8,alpha=0.4)
# FIGURE 3 — Angle 3: Constraint Violation Rate
fig3,ax3=plt.subplots(figsize=(9,4.5))
ax3.plot(tracker_pure.data["gen"],[v*100 for v in tracker_pure.data["viol_rate"]],"gray",linewidth=1.5,label="Pure NSGA-II")
ax3.plot(tracker_surr.data["gen"],[v*100 for v in tracker_surr.data["viol_rate"]],"red",linewidth=1.5,alpha=0.7,label="Pure Surrogate")
ax3.plot(tracker_refine.data["gen"],[v*100 for v in tracker_refine.data["viol_rate"]],"green",linewidth=2,label="Warm-Start Refined")
ax3.axhline(y=0,color="gray",ls=":",alpha=0.5)
ax3.set_xlabel("Generation");ax3.set_ylabel("Constraint Violation Rate [%]")
ax3.set_title("Angle 3: Constraint Violation Rate — Pure vs Surrogate vs Dual-Stage")
ax3.legend();ax3.grid(alpha=0.3)
plt.tight_layout();plt.savefig("angle3_violations.png",dpi=150);plt.show()
# FIGURE 4 — Angle 4: Pareto Front Comparison
# ============================================================
X_ps_all = res_surr_only.X
F_ps_phys = np.column_stack([X_ps_all[:,2]*X_ps_all[:,3], physics_outage(X_ps_all)])
fig4,ax4=plt.subplots(figsize=(9,6))
idx_b=np.argsort(res_pure.F[:,1]);Fb=res_pure.F[idx_b]
ax4.plot(Fb[:,1]*100,Fb[:,0],"o-",color="gray",markersize=3,linewidth=1.5,alpha=0.7,label=f"Baseline: Pure Physics ({len(Fb)} pts)")
idx_s=np.argsort(F_ps_phys[:,1]);Fps=F_ps_phys[idx_s]
ax4.plot(Fps[:,1]*100,Fps[:,0],"s--",color="red",markersize=3,linewidth=1.5,alpha=0.7,label=f"Pure Surrogate ({len(Fps)} pts)")
idx_r=np.argsort(res_refine.F[:,1]);Fr=res_refine.F[idx_r]
ax4.plot(Fr[:,1]*100,Fr[:,0],"D-",color="darkgreen",markersize=4,linewidth=1.8,label=f"Warm-Start Refined ({len(Fr)} pts)")
ax4.axvline(x=10,color="red",ls="--",alpha=0.5)
feas_b=Fb[:,1]<=0.10;feas_r=Fr[:,1]<=0.10
if feas_b.any():
    bi_b=np.argmin(Fb[feas_b,0]);ax4.plot(Fb[feas_b,1][bi_b]*100,Fb[feas_b,0][bi_b],"k*",markersize=16,label=f"Baseline Knee: {Fb[feas_b,0][bi_b]:.4f} m2")
if feas_r.any():
    bi_r=np.argmin(Fr[feas_r,0]);ax4.plot(Fr[feas_r,1][bi_r]*100,Fr[feas_r,0][bi_r],"g*",markersize=16,label=f"Warm-Start Knee: {Fr[feas_r,0][bi_r]:.4f} m2")
ax4.set_xlabel("Physics Outage [%]");ax4.set_ylabel("Area [m2]")
ax4.set_title("Angle 4: Pareto Front — Baseline vs Pure Surrogate vs Dual-Stage")
ax4.legend(fontsize=8);ax4.grid(alpha=0.3)
plt.tight_layout();plt.savefig("angle4_pareto.png",dpi=150);plt.show()

# Summary
print("\n"+"="*70)
print("{:30} {:12} {:25} {:15}".format("Method","Wall Clock","Knee (phys-verified)","Physics Evals"))
print("-"*70)
fp_feas=res_pure.F[:,1]<=0.10; fr_feas=res_refine.F[:,1]<=0.10; fps_feas=F_ps_phys[:,1]<=0.10
kp="{:.4f} m2".format(res_pure.F[fp_feas,0].min()) if fp_feas.any() else "N/A"
ks="{:.4f} m2".format(F_ps_phys[fps_feas,0].min()) if fps_feas.any() else "N/A"
kd="{:.4f} m2".format(res_refine.F[fr_feas,0].min()) if fr_feas.any() else "N/A"
print("{:30} {:12.0f}s {:25} {:15}".format("Pure Physics (NSGA-II 200g)",t_pure,kp,n_gen_pure*300))
print("{:30} {:12.0f}s {:25} {:15}".format("Pure Surrogate (LGBM 200g)",t_surr_only,ks,0))
print("{:30} {:12.0f}s {:25} {:15}".format("Dual-Stage (LGBM+Phys 20g)",t_surr_only+t_refine,kd,n_gen_warm*300))
print("="*70)
